# HunyuanOCR 1.5 on AMD Radeon W7900D

This notebook is an executable product demo, not a benchmark notebook. It combines an immutable ROCm/vLLM runtime, the exact HunyuanOCR-1.5 model payload, original synthetic documents, and an OpenAI-compatible OCR workflow.

**What you will see**

- one-click environment and model validation;
- an on-demand local vLLM service with mandatory `MIOPEN_FIND_MODE=2`;
- Chinese, English, mixed-layout, table, and low-resolution examples;
- rendered Markdown, HTML tables, and LaTeX;
- resumable batch inference, JSON/Markdown export, quality checks, and timing;
- an issue-ready diagnostics report.

> The first empty-cache model startup may take roughly 15 minutes. Subsequent starts reuse the mounted cache.


## 0. Demo Architecture

Jupyter is the container's main process. The service cell starts vLLM in the same container only when it is needed. Both processes use the same immutable model files, while generated caches and outputs remain under `/hunyuanOCR_workspace`.

```text
Browser -> JupyterLab :8892 -> Python kernel
                              |
                              +-> vLLM :18016 -> HunyuanOCR-1.5 -> W7900D
```

The notebook image is derived from:

```text
crpi-a7t9nblyxh55vyd2.cn-shanghai.personal.cr.aliyuncs.com/muzihao2/work@sha256:dc970503ebf22a87aab42a5c6cdb8b8cce5a42685c997b202e6ece6bdf3b1d86
```


In [ ]:
from pathlib import Path
import json
import os
import random
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import HTML, Markdown, display
from PIL import Image

ROOT = Path.cwd().resolve()
required_paths = (
    Path("src/hunyuanocr_demo.py"),
    Path("scripts/start_service.sh"),
    Path("assets/images"),
    Path("outputs"),
)
if not all(path.exists() for path in required_paths):
    raise RuntimeError(
        "Start this notebook from its bundle directory. Keep "
        "hunyuan_ocr_demo.ipynb, src, scripts, assets, and outputs together."
    )
os.environ["HUNYUANOCR_DEMO_ROOT"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.hunyuanocr_demo import (
    ASSETS_DIR,
    BASE_IMAGE_DIGEST,
    DOC_PARSE_PROMPT,
    MODEL_DIR,
    OUTPUT_DIR,
    archive_outputs,
    batch_infer,
    diagnostic_report,
    ensure_layout,
    infer_image,
    inspect_runtime,
    list_images,
    performance_summary,
    runtime_report,
    save_result,
    service_status,
    start_service,
    stop_service,
    summary_rows,
    tail_service_log,
    write_markdown_report,
)
from src.visualize import visualize_result

random.seed(20260828)
paths = ensure_layout()
paths


## 1. Verify the Immutable Runtime

The checks below report the actual container, Python, ROCm/PyTorch, visible GPU, model payload, cache, and output permissions. Files supplied by the mapped notebook bundle are resolved from sibling `src`, `scripts`, `assets`, and `outputs` paths. Image-owned model, runtime, Jupyter, and cache resources remain under the fixed `/hunyuanOCR_workspace` root.


In [ ]:
report = runtime_report()
assert report["base_image_digest"] == BASE_IMAGE_DIGEST
assert report["miopen_find_mode"] == "2"
assert report["paths"]["workspace_exists"] is True
assert report["paths"]["bundle_root"] == str(ROOT)
assert report["paths"]["bundle_layout_complete"] is True
assert report["paths"]["model_weight_exists"] is True
assert report["paths"]["model_weight_bytes"] == 2_239_932_512
assert report["paths"]["output_writable"] is True
assert report["torch_probe"]["returncode"] == 0
assert report["torch_probe"]["json"]["cuda_available"] is True
assert report["torch_probe"]["json"]["device_count"] == 1
report


## 2. Inspect HunyuanOCR Entrypoints

The demo inspects the image rather than guessing an inference API. It verifies the embedded model manifest, vLLM executable, supported Python modules, exact model name, and service parameters. The production entrypoint remains the authority for GPU and model validation.


In [ ]:
inspection = inspect_runtime()
assert inspection["entrypoint"] == "/hunyuanOCR_workspace/bin/entrypoint.sh"
assert inspection["model_file_count"] == 24
assert inspection["configuration"]["model"] == "tencent/HunyuanOCR"
assert inspection["configuration"]["miopen_find_mode"] == "2"
assert inspection["configuration"]["attention_backend"] in {"ROCM_ATTN", "TRITON_ATTN"}
display(pd.DataFrame(inspection["model_files"]).sort_values("bytes", ascending=False).head(10))
inspection["configuration"]


## 3. Start the OCR Service

This cell starts the validated autoregressive vLLM server with BF16, TP=1, context length 131072, and `MIOPEN_FIND_MODE=2`. The decoder uses the `VLLM_ATTENTION_BACKEND` selected when the container was created (`ROCM_ATTN` by default, or `TRITON_ATTN`); the vision encoder remains on `TORCH_SDPA`. Stop and recreate the container to switch backends. Missing weights, an unavailable GPU, an incorrect architecture, or a path mismatch produces a clear exception with the server log tail.


In [ ]:
startup = start_service(timeout_seconds=1800)
assert startup["ready"] is True
assert startup["attention_backend"] == inspection["configuration"]["attention_backend"]
model_info = startup["models"]["data"][0]
assert model_info["id"] == "tencent/HunyuanOCR"
assert model_info["root"] == "/hunyuanOCR_workspace/models/HunyuanOCR"
assert model_info["max_model_len"] == 131072
startup


## 4. Synthetic Demo Documents

All samples are original, reproducible images generated by `scripts/generate_assets.py`; no benchmark scans are redistributed. They cover a bilingual two-column page, an invoice-style HTML table, and a rotated low-resolution note.


In [ ]:
images = list_images()
assert len(images) == 3, images
fig, axes = plt.subplots(1, len(images), figsize=(18, 8))
for axis, path in zip(axes, images):
    with Image.open(path) as image:
        axis.imshow(image)
        axis.set_title(f"{path.name}\n{image.width} x {image.height}")
    axis.axis("off")
plt.tight_layout()
plt.show()


## 5. Single-Page OCR

This cell is a self-contained input-to-output example: it displays the original page, starts or reuses the local vLLM service from inside this notebook, runs HunyuanOCR, and renders the extracted Markdown below. No terminal command is required. The first empty-cache startup can take roughly 15 minutes; later runs reuse the mounted cache.


In [ ]:
single_path = ASSETS_DIR / "01_bilingual_columns.png"
display(Markdown("### Input: original document"))
with Image.open(single_path) as source_image:
    display(source_image.copy())

if not service_status()["ready"]:
    display(Markdown("**Starting HunyuanOCR inside this notebook...**"))
    start_service(timeout_seconds=1800)

single_result = infer_image(single_path)
assert single_result["quality"]["passed"] is True
assert single_result["finish_reason"] == "stop"
saved = save_result(single_result)
display(Markdown("### Output: extracted text"))
display(Markdown(single_result["text"]))
print(json.dumps({
    "status": single_result["status"],
    "timing": single_result["timing"],
    "usage": single_result["usage"],
    "saved": {key: str(value) for key, value in saved.items()},
}, ensure_ascii=False, indent=2))


## 6. Render Markdown, HTML, and LaTeX

HunyuanOCR `doc_parse` returns reading-order Markdown. Jupyter renders headings, tables, and LaTeX directly. The panel below places the source page and rendered OCR text side by side.


In [ ]:
visual_path = OUTPUT_DIR / "visualizations" / "01_bilingual_columns_result.png"
visualize_result(single_path, single_result, visual_path)
with Image.open(visual_path) as image:
    display(image)
print(f"Saved: {visual_path}")


### About text boxes and confidence

The validated `doc_parse` endpoint returns Markdown in reading order; it does **not** return calibrated bounding boxes or confidence scores. The demo therefore does not invent them. The normalized `blocks` array exposes `bbox=null` and `confidence=null`. A later notebook can add a separately validated spotting/layout task if spatial output is required.


In [ ]:
pd.DataFrame(single_result["blocks"])[["index", "type", "bbox", "confidence", "text"]]


## 7. Complex Layout and Table

The invoice sample exercises mixed Chinese/English text, numeric values, currency symbols, table structure, and reading order. HTML emitted by the model is rendered as part of the returned Markdown.


In [ ]:
invoice_path = ASSETS_DIR / "02_invoice_table.png"
invoice_result = infer_image(invoice_path)
assert invoice_result["quality"]["passed"] is True
save_result(invoice_result)
display(Markdown(invoice_result["text"]))
invoice_result["timing"], invoice_result["usage"]


## 8. Batch OCR

The batch runner filters image extensions, catches per-file exceptions, writes each result immediately, and resumes from existing JSON. Set `resume=False` to deliberately re-run every sample.


In [ ]:
batch_results = batch_infer(ASSETS_DIR, OUTPUT_DIR, resume=True)
rows = summary_rows(batch_results)
summary = pd.DataFrame(rows)
display(summary)
assert len(summary) == 3
assert summary["quality_pass"].all()
assert (summary["status"] == "PASS").all()


## 9. Export Results

Each page has stable JSON and Markdown output. The combined report is human-readable, and the ZIP archive preserves all generated evidence for review or downstream ingestion.


In [ ]:
report_path = write_markdown_report(batch_results)
archive_path = archive_outputs()
reloaded = json.loads((OUTPUT_DIR / "json" / "01_bilingual_columns.json").read_text())
assert reloaded["quality"]["passed"] is True
assert report_path.is_file() and archive_path.is_file()
print(f"Markdown report: {report_path}")
print(f"Archive: {archive_path} ({archive_path.stat().st_size:,} bytes)")


## 10. Performance and GPU Memory

The table separates service startup from request latency. Request timing uses the completed HTTP response, while the server owns GPU synchronization. The plot relates submitted megapixels to latency. This three-page demo is descriptive, not an OmniDocBench score.


In [ ]:
performance = performance_summary(batch_results)
display(performance)
plot_data = pd.DataFrame(summary_rows(batch_results)).dropna(subset=["megapixels", "latency_seconds"])
axis = plot_data.plot.scatter(x="megapixels", y="latency_seconds", s=90, color="#0f766e", figsize=(8, 5), title="OCR latency vs input size")
for _, row in plot_data.iterrows():
    axis.annotate(row["file"].split("_")[0], (row["megapixels"], row["latency_seconds"]), xytext=(5, 4), textcoords="offset points")
plt.grid(alpha=0.25)
plt.show()
report["rocm_smi"]["stdout"][:2000]


## 11. Quality Gate

HTTP success is not enough. Every output is checked for empty text, coordinate-only collapse, and suspiciously short content. This guard exists because silent numerical corruption can otherwise look like a successful OCR request.


In [ ]:
quality_table = pd.DataFrame([
    {"file": result["source"]["name"], **result["quality"]}
    for result in batch_results
])
display(quality_table)
assert quality_table["passed"].all()


## 12. Reproducibility and Diagnostics

The final cell writes an issue-ready report containing package versions, image digest, GPU/runtime facts, model inventory, service status, configuration, log tail, and common failure guidance for OOM, missing fonts, corrupt images, wrong model paths, or import failures.


In [ ]:
diagnostics = diagnostic_report()
diagnostic_path = OUTPUT_DIR / "diagnostics.json"
assert diagnostic_path.is_file()
assert diagnostics["runtime"]["base_image_digest"] == BASE_IMAGE_DIGEST
assert diagnostics["service"]["ready"] is True
print(json.dumps({
    "diagnostics": str(diagnostic_path),
    "base_image_digest": diagnostics["runtime"]["base_image_digest"],
    "model_dir": diagnostics["inspection"]["model_dir"],
    "service_ready": diagnostics["service"]["ready"],
}, indent=2))


## Optional: Stop the OCR Service

Jupyter can remain open while vLLM is stopped to release GPU memory. Run the next cell only when the interactive OCR portion is finished. `make stop` outside the notebook stops the complete container.


In [ ]:
# Uncomment to release GPU memory while keeping JupyterLab running.
# stop_service()
service_status()
